In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key = "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"
)
response = llm.invoke("Who are you")
print(response.content)

I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."


In [4]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://amazon.jobs/en/jobs/10408884/business-intelligence-engineer-worldwide-engineering-and-innovation-north-american-sortation-and-ultrafast-engineering")
page_data = loader.load().pop().page_content
print(page_data)

USER_AGENT environment variable not set, consider setting it to identify your requests.


Business Intelligence Engineer , Worldwide Engineering and Innovation North American Sortation and Ultrafast Engineering - Job ID: 10408884 | Amazon.jobs
Skip to main content×HomeTeamsLocationsJob categoriesMy careerMy applicationsMy profileAccount securitySettingsSign outResourcesAccommodationsBenefitsInclusive experiencesHow We HireLeadership principlesWorking at AmazonFAQBusiness Intelligence Engineer , Worldwide Engineering and Innovation North American Sortation and Ultrafast EngineeringJob ID: 10408884 | Amazon.com Services LLCApply nowDescriptionThe Business Intelligence Engineer (BIE) will be part of a high-visibility, growth in a dynamic, entrepreneurial NASC Engineering team focused on innovation and automation initiatives that supports the North American Sort Centers. We are seeking an exceptionally talented and deeply focused BI Engineer with focused analytical skills who will build tools and reporting for senior leadership. The BIE will contribute to business outcomes by a

In [5]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    
        '''
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        '''
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
print(res.content)

```json
{
  "role": "Business Intelligence Engineer",
  "experience": {
    "years": 3,
    "requirements": [
      "3+ years of analyzing and interpreting data with Redshift, Oracle, NoSQL etc. experience",
      "1+ years of SQL, ETL or Oracle experience",
      "1+ years of processing large, multi-dimensional datasets from multiple sources experience",
      "1+ years of performing statistical analysis experience",
      "1+ years of developing automated reporting experience"
    ]
  },
  "skills": [
    "Data visualization using Tableau, Quicksight, or similar tools",
    "Data modeling, warehousing and building ETL pipelines",
    "Statistical Analysis packages such as R, SAS and Matlab",
    "SQL to pull data from a database or data warehouse and scripting experience (Python) to process data for modeling",
    "AWS solutions such as EC2, DynamoDB, S3, and Redshift",
    "Data mining, ETL, etc. and using databases in a business environment with large-scale, complex datasets"
  ],


In [11]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
print(json_res)

{'role': 'Business Intelligence Engineer', 'experience': {'years': 3, 'requirements': ['3+ years of analyzing and interpreting data with Redshift, Oracle, NoSQL etc. experience', '1+ years of SQL, ETL or Oracle experience', '1+ years of processing large, multi-dimensional datasets from multiple sources experience', '1+ years of performing statistical analysis experience', '1+ years of developing automated reporting experience']}, 'skills': ['Data visualization using Tableau, Quicksight, or similar tools', 'Data modeling, warehousing and building ETL pipelines', 'Statistical Analysis packages such as R, SAS and Matlab', 'SQL to pull data from a database or data warehouse and scripting experience (Python) to process data for modeling', 'AWS solutions such as EC2, DynamoDB, S3, and Redshift', 'Data mining, ETL, etc. and using databases in a business environment with large-scale, complex datasets'], 'description': 'The Business Intelligence Engineer (BIE) will be part of a high-visibility,

In [12]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [25]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [24]:
job = json_res

In [26]:
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
links


[[{'links': 'https://example.com/flutter-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/typescript-frontend-portfolio'}],
 [{'links': 'https://example.com/python-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/react-portfolio'},
  {'links': 'https://example.com/flutter-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}]]

In [27]:
job = json_res
job['skills']


['Data visualization using Tableau, Quicksight, or similar tools',
 'Data modeling, warehousing and building ETL pipelines',
 'Statistical Analysis packages such as R, SAS and Matlab',
 'SQL to pull data from a database or data warehouse and scripting experience (Python) to process data for modeling',
 'AWS solutions such as EC2, DynamoDB, S3, and Redshift',
 'Data mining, ETL, etc. and using databases in a business environment with large-scale, complex datasets']

In [30]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at ABC. ABC is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of ABC 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase ABC's portfolio: {link_list}
        Remember you are Mohan, BDE at ABC. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Business Intelligence Engineering Solutions for NASC Engineering Team

Dear Hiring Manager,

I came across the job description for a Business Intelligence Engineer at your organization and was impressed by the exciting initiatives your team is undertaking. As a Business Development Executive at ABC, an AI & Software Consulting company, I believe our expertise can help you achieve your goals.

Our team at ABC has extensive experience in developing and implementing tailored solutions that drive scalability, process optimization, cost reduction, and heightened overall efficiency. We have empowered numerous enterprises with our automated tools, and I'd like to highlight a few relevant examples from our portfolio:

* Our work in machine learning and Python can be seen at https://example.com/ml-python-portfolio, where we've developed predictive models and data analytics solutions for various clients.
* Our expertise in DevOps is showcased at https://example.com/devops-portfol